# Heritage Short — Stage 3: Hero Moving Shots with Wan 2.2 TI2V-5B

Run on Kaggle with **Accelerator: GPU T4 x2** (or T4 / P100) and **Internet: ON**.
1. In the Kaggle right sidebar, click **+ Add Input -> Upload -> New Dataset** and upload `outputs/keyframes/` PNGs (at minimum `shot_01.png`, `shot_09.png`, `shot_11.png`). Any dataset name works — this notebook searches `/kaggle/input/**` automatically.
2. `storyboard.json` is automatically read from your Kaggle dataset or downloaded from `https://raw.githubusercontent.com/Taddeal/tenx/main/storyboard.json`.
3. Uses HuggingFace `diffusers` (`WanImageToVideoPipeline` + `AutoencoderKLWan` with `enable_model_cpu_offload`) so there is no `flash-attn` compilation failure on Kaggle T4.
4. Download `00_shots_wan.zip` and `00_wan_runs.jsonl` from `/kaggle/working/` and extract the `.mp4` files into `outputs/shots/wan/` locally.

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece protobuf imageio imageio-ffmpeg hf_transfer

In [ ]:
import glob
import json
import os
import shutil
import time
import urllib.request
import torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

OUT_DIR = '/kaggle/working/shots_wan'
os.makedirs(OUT_DIR, exist_ok=True)

# 1. Locate keyframes anywhere under /kaggle/input/ regardless of dataset name
keyframe_map = {}
for p in glob.glob('/kaggle/input/**/*.png', recursive=True):
    name = os.path.basename(p)
    if name.startswith('shot_') and name.endswith('.png'):
        keyframe_map[name] = p

if not keyframe_map:
    raise FileNotFoundError(
        'No shot_*.png keyframes found under /kaggle/input/. '
        'Upload outputs/keyframes/*.png via + Add Input -> Upload -> New Dataset in Kaggle first.'
    )
print(f'Found {len(keyframe_map)} keyframes under /kaggle/input/: {sorted(keyframe_map.keys())}')

# 2. Locate storyboard.json in /kaggle/input/ or fetch directly from GitHub
sb_candidates = glob.glob('/kaggle/input/**/storyboard.json', recursive=True)
if sb_candidates:
    sb_path = sb_candidates[0]
    print(f'Using storyboard from dataset: {sb_path}')
else:
    sb_path = '/kaggle/working/storyboard.json'
    url = 'https://raw.githubusercontent.com/Taddeal/tenx/main/storyboard.json'
    print(f'Downloading storyboard.json from {url}...')
    urllib.request.urlretrieve(url, sb_path)

with open(sb_path, 'r', encoding='utf-8') as f:
    sb = json.load(f)

# 3. Load Wan2.2-TI2V-5B via Diffusers with CPU offload for Kaggle 16GB T4
MODEL_ID = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
print(f'Loading {MODEL_ID}...')
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
if hasattr(pipe.vae, 'enable_tiling'):
    pipe.vae.enable_tiling()

# Hero draft settings tuned for Kaggle T4 speed (height/width divisible by 32; (num_frames-1)%4 == 0)
HEIGHT, WIDTH = 480, 832
NUM_FRAMES = 49
STEPS = 25
PRIORITY_SHOTS = [1, 9, 11]
NEG_PROMPT = 'Bright tones, overexposed, static, blurred details, subtitles, watermark, worst quality, low quality, warped face'

logs = []
for shot in sb['shots']:
    sid = shot['id']
    if shot.get('is_end_card') or sid not in PRIORITY_SHOTS:
        continue
    img_name = f'shot_{sid:02d}.png'
    img_path = keyframe_map.get(img_name)
    if not img_path:
        print(f'SKIP {img_name}: not present in /kaggle/input/')
        continue
    out_mp4 = os.path.join(OUT_DIR, f'shot_{sid:02d}.mp4')
    if os.path.exists(out_mp4):
        continue

    t0 = time.time()
    status = 'success'
    err = None
    try:
        image = load_image(img_path).convert('RGB').resize((WIDTH, HEIGHT))
        gen = torch.Generator(device='cpu').manual_seed(int(shot.get('seed', 1000 + sid)))
        print(f'Generating {img_name}: {shot["motion_prompt"]}')
        frames = pipe(
            image=image,
            prompt=shot['motion_prompt'],
            negative_prompt=NEG_PROMPT,
            height=HEIGHT,
            width=WIDTH,
            num_frames=NUM_FRAMES,
            guidance_scale=5.0,
            num_inference_steps=STEPS,
            generator=gen,
        ).frames[0]
        export_to_video(frames, out_mp4, fps=24)
    except Exception as exc:
        status = 'error'
        err = str(exc)[:280]
        print(f'FAIL shot {sid:02d}: {err}')

    elapsed = round(time.time() - t0, 2)
    logs.append({
        'stage': 'shots',
        'model': MODEL_ID,
        'shot_id': sid,
        'size': f'{WIDTH}x{HEIGHT}',
        'num_frames': NUM_FRAMES,
        'steps': STEPS,
        'seed': shot.get('seed'),
        'latency_seconds': elapsed,
        'gpu_minutes': round(elapsed / 60, 2),
        'status': status,
        'error': err,
    })
    print(f'Shot {sid:02d} -> {status} in {elapsed}s')

for log_path in ['/kaggle/working/wan_runs.jsonl', '/kaggle/working/00_wan_runs.jsonl']:
    with open(log_path, 'w') as f:
        for row in logs:
            f.write(json.dumps(row) + '\n')

shutil.make_archive('/kaggle/working/00_shots_wan', 'zip', '/kaggle/working', 'shots_wan')
print('Done! Download 00_shots_wan.zip and 00_wan_runs.jsonl from /kaggle/working/')